DATA PIPELINE CON TF.DATA: GESTIONE PROFESSIONALE DEL FLUSSO DI DATI

tf.data è il modo professionale di TensorFlow per alimentare un modello senza buttare tutti i dati dentro model.fit in modo grezzo.
Alimentiamo il modello in modo efficiente, senza mandare in crisi il computer anche quando i dati sono tantissimi.
Il modello non deve solo imparare, deve ricevere dati puliti, trasformati, mescolati, divisi in batch e caricati abbastanza velocemente. Se la pipeline è lenta, CPU/disco fanno da collo di bottiglia e la GPU resta ferma. TensorFlow descrive tf.data proprio come API per creare pipeline efficienti e complesse da fonti diverse.

Con l'aumentare delle dimensioni dei dataset, caricare i dati interamente in memoriza con NumPy porta, inevitabilmente, alla saturazione della RAM e rallentamenti critici.
Il modulo tf.data introduce l'oggetto 'Dataset', una potente astrazione che permette di rappresentare sequenze di elementi potenzialmente infinite, gestendo il caricamento solo quando necessario. Questo flusso prende il nome di dataset.

- Il metodo 'from_tensor_slice è il punto di ingresso più comune, accetta array NumPy o tensori e li scompone i elementi individuali lungo la prima dimensione. Scompone gli array in elementi
- L'astrazione del dataset permette di incapsulare diverse sorgenti di dati, come file CSV, immagini su disco o record binari, in un'unica interfaccia coerente.
- Un oggetto 'Dataset' è immutabile: ogni trasformazione applicata restituisce un nuovo oggetto, permettendo di costruire catene di operazioni leggibili ed efficienti.
- La creazione del dataset definisce la firma dei dati che fluiranno verso il modello.
Una volta definita la sorgente dobbiamo definire la logica della consegna.

Per addestrare bene una rete, non basta darlgi i dati, bisogna darli in modo coerente.
Con il BATCHING  l'operazione di 'batch' raggruppa gli elementi individuali in blocchi di dimensione fissa, pronti per essere processati in parallelo dalla GPU, massimizzando l'efficienza.
Con lo shuffling invece mescoliamo il mazzo ad ogni epoca, è fondamentale per evitare che il modello impari l'ordine dei dati.
Con il repeat trasformiamo un dataset limitato  in un flusso continuo attraverso molteplici epoche di addestramento.

La gestione della RAM
Lazy Evaluation
A differenza degli array NumPy, td.data.dataset, segue la logica dela valutazione prigra: gli elementi non vengono caricati finchè il modello non li richiede esplicitamente.
Questo permette di gestire dataset che superano di gran lunga la capacità della memoria fisica, caricando solo il mini-batch corrente e scartando quello precedente.
Il tf.data.dataset non carica i dati finchè il modello dice, mi serve il nuovo batch, questo permette di avere dataset che sono 10, 100 dataset che sono molto di più della memoria a disposizione.

Eliminare i colli di bottiglia tra CPU e GPU
l'addestramento di una rete neurale è spesso rallentato dai tempi di attesa delle GPU che aspetta che la CPU prepari il prossimo batch di dati.
E' possibile trasformare questo processo sequenziale in una pipeline parallela altamente performante utilizzando metodi specifici di tf.data

Massimizzare il throughput
Il metodo 'prefetch' sovrappone il lavoro del produttore (CPU) a quello del consumatore (GPU), preparando il batch successivo mentre quello attuale è in elaborazione
- L'uso di tf.data.AUTOTUNE permette a TensorFlow di decidere dinamicamente il numero ottimale di elementi da pre-caricare in base all'hardware disponibile.
- Il metodo 'cache' salva il dataset trasformato in memoria o su disco locale dopo la prima epoca, evitando di ricalcolare le trasformazioni costose nei cicli successivi.
- L'obbiettivo è ridurre il tempo totale di addestramento minimizzando i tempi morti dei processori.

Ottimizzare significa far parlare la stessa lingua a software e hardware, usando il parallelismo, possiamo istruire TensorFlow a usare tutti i core della CPU per deprimere o trasformare i file simultaneamente.
Il BUFFER agisce come un serbatoio: se la CPU ha un rallentamente temporaneo, la GPU può continuare a pescare dati dal prefetch senza fermarsi.
Il chaching su disce è particolarmente utile per dataset remoti o vie rete, portando i dati vicino al calcolo dopo il primo accesso.

Senza prefetch, senza ottimizzare, la GPU rimane ferma durante il caricamente dei dati. Con prefetch, le due attività si sovrappongono quasi perfettamente nel tempo.
Questo cambio di paradisma può ridurre il tempo di addestramento globale anche del 50% su dataset complessi con molto preprocessing.

Ma oltre a caricare i dati, dobbiamo anche trasformarli
Mapping e Preprocessing
Trasformazione vettoriali on-the-fly
Ilterzo pilastro di una pipelina moderna è la capacità di trasformare i dati mentre fluiscono verso il modello, applicando correzioni, normalizzazioni o aumenti di dati.
Il metodo MAP è lo strumento principale per applicare funzioni arbitrarie ad ogni elemento del dataset in modo estremamente efficiente e scalabile.

Il potere di MAP
- La funzione 'map' applica una trasformazione specifica a ogni campione, permettendo di gestire compiti come il ridimensionamento delle immagini o il filtraggio numerico.
- Possiamo parallelizzare l'esecuzione del mapping impostando num_parallel_call,  accelerando drasticamente il preprocessing intensivo.
- Il mapping trasforma la struttura degli elementi del dataset mantenendo la fluidità della pipeline.

Efficienza del Mapping
Se possibile, è preferibile applicare le trasformazoini dopo il batching per sfruttare le istruzioni vettoriali della CPU su più dati contemporaneamente.
Tramite tf.cond o td.where possiamo applicare logiche diverse a campioni differenti all'interno della setssa funzione di mapping.
Il mapping è il luogo ideale per iniettare rumore gaussiano o effettuare rotazioni casuali durante l'addestramento per miglioare la robustezza.

Un buon ingegnere del deep, separe chiaramente la logica del modello dalla logica della pipeline di dati, rendendo i due compnenti testabili indipendentemente.
Utilizzare le trasformazioni all'interno di tf.data garantisce la logica del mo dello dalla logica della pipeline di dati.

Schema base:

dataset = tf.data.Dataset.from_tensor_slices((X, y))

dataset = dataset.shuffle(buffer_size=1000)
dataset = dataset.batch(32)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

from_tensor_slice prende i dati riga per riga
shuffle mescola le righe. Serve soprattutto per training, perchè il modello non deve imparare l'ordine del file
batch(32) da al modello 32 recordo alla volta, non uno per uno.
prefetch(td.data.AUTOTUNE) prepara il batch successivo mentre il modello sta lavorando sul batch corrente. Questa è una delle ottimizzazioni principali di TernsorFlow. Elimina o riduce i tempi morti

Esempio: 
import tensorflow as tf
import numpy as np

X = np.array([
    [10.0, 2.0],
    [20.0, 4.0],
    [30.0, 6.0],
    [40.0, 8.0]
])

y = np.array([1.2, 2.4, 3.6, 4.8])

dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(4).batch(2).prefetch(tf.data.AUTOTUNE)

for batch_X, batch_y in dataset:
    print(batch_X)
    print(batch_y)

tf.data non migliora il modello, migliora il flusso di dati. Se i dati ERP sono sporchi, pesi sbagliati, materiali non codificati bene, ecc, il modello imparerà spazzatura con grande efficienza
Ordine consigliato:
1. prepari dataset puliti in pandas
2. separi X e y
3. normalizzi i numeri
4. trasformi in tf.data.dataset
5. applichi shuffle, batch, prefetch
6. alleni il modello

In [3]:
import tensorflow as tf
import numpy as np

# 1. CREAZIONE DEL'OGGETTO TD.DATASET 
#Simuliamo 10.000 campioni con 20 feature ciascuno
X_raw=np.random.uniform(0,255,(10000,20)).astype(np.float32)
y_raw= np.random.randint(0,2,(10000,1)).astype(np.float32)

#Trasformiamo gli array NumPy in un oggetto Dataset
# from_tensor_slices scompone gli array lungo la prima dimensione, creando coppie (X,y) per ogni campione.
dataset=tf.data.Dataset.from_tensor_slices((X_raw,y_raw))

#2. MAPPING E PREPORCESSING
def normalize_and_clean(feature, label):
    """
    Funzione di preprocessing che verrà applicata a ogni campione del dataset. Normalizza le feature e pulisce le etichette.
    """
    feature = feature/255.0  #Normalizzazione delle feature portandole tra 0 e 1
    label = tf.cast(label, tf.float32)
    return feature, label
#Applichiamo la funzione di preprocessing a ogni campione del dataset usando map
#num_parallel_calls=tf.data.AUTOTUNE permette di parallelizzare l'operazione di mapping, migliorando le prestazioni.
dataset=dataset.map(normalize_and_clean,num_parallel_calls=tf.data.AUTOTUNE)

#3. OTTIMIZZAZOINE PERFORMANCE
#Cache: salva i dati processati in RAM dopo la prima epoca per non rifare i calcoli.
dataset=dataset.cache()

#Shuffle & Batching
dataset=dataset.shuffle(buffer_size=1000).batch(32)

#Prepetch: la CPU prepare il batch mentre la GPU elabora il batch n
#Fondamentale  per eliminare i colli di bottiglia tra CPU e GPU, migliorando l'efficienze del training.
dataset=dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

#TEST DEL DATASET
model=tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

#Ora passiamo direttamente il dataset al modello.fit
model.fit(dataset, epochs=5)





Epoch 1/5


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4995 - loss: 0.6960
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5008 - loss: 0.6942
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5071 - loss: 0.6935
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5109 - loss: 0.6931
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5151 - loss: 0.6926
